# Stable Audio Open 1.0

https://huggingface.co/stabilityai/stable-audio-open-1.0




In [ ]:

!pip uninstall -y torchao
!pip install -q -U torchsde diffusers soundfile

import os
import torch
import soundfile as sf
from diffusers import StableAudioPipeline
from huggingface_hub import login
from IPython.display import Audio, display

# Sign in to Hugging Face
login()

# Detect device and set appropriate dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

# Load the pretrained audio generation model
pipe = StableAudioPipeline.from_pretrained(
    "stabilityai/stable-audio-open-1.0",
    dtype=torch_dtype
).to(device)

# Describe the fictional animal sound
prompt = (
    "A fictional zebra-like animal making strange rhythmic calls, "
    "a low rumble followed by short breathy chirps, natural animal "
    "sound effects, no speech, no music"
)

negative_prompt = "Low quality, distorted audio, human speech, music"

generator = torch.Generator(device).manual_seed(42) if device == "cuda" else torch.Generator().manual_seed(42)

# Generate one 5-second sound
audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    audio_end_in_s=5.0,
    num_waveforms_per_prompt=1,
    generator=generator,
).audios

# Save and play the sound
sr = pipe.vae.sampling_rate
os.makedirs("fictional_animal_sounds", exist_ok=True)

for i, sample in enumerate(audio):
    output_path = f"fictional_animal_sounds/test_animal_{i}.wav"
    sf.write(output_path, sample.T.float().cpu().numpy(), sr)
    display(Audio(output_path))

print("Test sound generated and saved successfully.")

# Stable Audio 3 — Small-Music

Text-to-music with Stability AI’s `stable-audio-3-small-music` (433M parameters, up to 120 s, stereo 44.1 kHz). It runs on CPU, so no GPU runtime is needed (a GPU makes it faster).

**Before running:**
1. Log in to Hugging Face and accept the terms on the [model page](https://huggingface.co/stabilityai/stable-audio-3-small-music). It’s gated, and the terms include Gemma’s (the text encoder is T5Gemma).
2. Create a read token (huggingface.co → Settings → Access Tokens) and add it as a Colab secret named `HF_TOKEN`, or paste it when `login()` asks.
3. Run the install cell. If Colab asks you to **restart the session**, do it, then continue from the next cell.

In [ ]:
# Install the Stable Audio 3 library (may replace Colab's torch with the version it pins)
!pip install -q "git+https://github.com/Stability-AI/stable-audio-3.git" soundfile

In [ ]:
import torch
import soundfile as sf
from huggingface_hub import login
from IPython.display import Audio, display
from stable_audio_3 import StableAudioModel

login()  # skip if HF_TOKEN is set in Colab secrets

device = "cuda" if torch.cuda.is_available() else "cpu"
model = StableAudioModel.from_pretrained("small-music", device=device)

In [ ]:
prompt = (
    "Raw wildlife sound recording of a large unknown animal roaring. "
    "One deep guttural throat roar, followed by two shorter rough snarls. "
    "Animal vocalization only, isolated dry sound, background ambience."
    )

out = model.generate(
    prompt=prompt,
    duration=21,    # seconds (max 120 for small-music)
    steps=8,        # default; more steps don't help the post-trained model
    seed=0,         # -1 = random
    batch_size=3,   # three variations
)
# Note: cfg_scale and negative_prompt only affect the “-base” checkpoints.

In [ ]:
# Normalize the output to a (batch, channels, samples) tensor plus sample rate
def unpack(out, default_sr=44100):
    if isinstance(out, (tuple, list)) and len(out) == 2:
        a, b = out
        if isinstance(a, int):
            return b, a
        if isinstance(b, int):
            return a, b
    return out, default_sr

audio, sr = unpack(out)
audio = torch.as_tensor(audio).float().cpu()
if audio.dim() == 2:          # single clip: (channels, samples)
    audio = audio.unsqueeze(0)

for i, clip in enumerate(audio):
    clip = clip / clip.abs().max().clamp(min=1e-8)   # peak normalize
    path = f"sa3_small_music_{i}.wav"
    sf.write(path, clip.T.numpy(), sr)
    print(path, f"{clip.shape[-1] / sr:.1f} s @ {sr} Hz")
    display(Audio(path))

**16 Animal Sounds**



In [ ]:
import os
import torch
import soundfile as sf
from IPython.display import Audio, display

# Create folder for generated sounds
os.makedirs("sounds", exist_ok=True)

# 16 fictional animal sounds with different prompts and durations
animals = [
    ("01_crystal_roar",
     "A fictional giant animal producing a deep, cracked roar that ends in a vibrating growl. Isolated raw wildlife vocalization, no music.",
     6),

    ("02_moon_howler",
     "An imaginary nocturnal creature making a long, wavering howl that breaks into two short cries. Dry animal recording, no music.",
     8),

    ("03_tiny_squeaker",
     "A tiny invented animal making irregular squeaks, quick chirps and little breathy peeps. Close-up animal vocalizations only, no music.",
     5),

    ("04_stoneback_grunts",
     "A heavy, fictional beast breathing loudly and producing several deep throaty grunts. Natural animal sounds only, no music.",
     7),

    ("05_rattlejaw",
     "An imaginary creature making a sequence of dry clicking sounds followed by a fast rattling hiss. Isolated animal sound effect, no music.",
     6),

    ("06_marsh_bellow",
     "A strange swamp-dwelling animal releasing two low, hollow bellows with a short pause between them. Raw animal vocalization, no music.",
     10),

    ("07_glass_whistler",
     "A fictional long-beaked creature making thin, breathy whistles that alternate between high and low pitches. Animal calls only, no music.",
     5),

    ("08_nose_snuffler",
     "A large imaginary mammal snorting, sniffing and forcefully exhaling several times near the microphone. Natural animal recording, no music.",
     6),

    ("09_bubble_throat",
     "An invented amphibious creature producing wet gurgles, bubbling throat noises and a final low croak. Animal sounds only, no music.",
     8),

    ("10_night_clicker",
     "A small fictional predator making rapid dry clicks, short chattering bursts and one low throaty call. Isolated wildlife recording, no music.",
     5),

    ("11_ember_growler",
     "A fictional animal producing a continuous rough, raspy growl that rises in pitch and ends with a short snarl. No music.",
     7),

    ("12_feather_triller",
     "An imaginary bird-like animal producing a rapid rolling trill, uneven chirps and a few sharp squeals. Natural animal calls only, no music.",
     6),

    ("13_cave_screecher",
     "A strange cave-dwelling creature giving one harsh, raspy screech followed by several quieter croaking calls. Animal vocalization only, no music.",
     5),

    ("14_thunderjaw",
     "A huge fictional animal making three separate deep roars, each rough and breathy, with silence between calls. Raw animal sound effect, no music.",
     9),

    ("15_moss_puffer",
     "A small round imaginary forest creature making soft huffs, nasal snuffles and a brief high-pitched squeal. Close animal recording, no music.",
     6),

    ("16_distant_echoer",
     "A mysterious fictional animal producing a low resonant call followed by a descending, rough howl. Isolated creature vocalization, no music.",
     10),
]


# Convert model output into audio tensor and sample rate
def unpack(out, default_sr=44100):
    if isinstance(out, (tuple, list)) and len(out) == 2:
        a, b = out
        if isinstance(a, int):
            return b, a
        if isinstance(b, int):
            return a, b

    return out, default_sr


# Generate and save all 16 sounds
for name, prompt, duration in animals:

    print(f"\nGenerating {name} ({duration} seconds)...")

    try:
        out = model.generate(
            prompt=prompt,
            duration=duration,
            steps=8,
            seed=42 + int(name[:2]),
            batch_size=1,
        )

        audio, sr = unpack(out)
        audio = torch.as_tensor(audio).float().cpu()

        # Ensure the audio has a batch dimension
        if audio.dim() == 2:
            audio = audio.unsqueeze(0)

        # Save the first generated clip
        clip = audio[0]

        # Normalize volume safely
        clip = clip / clip.abs().max().clamp(min=1e-8)

        # Save as WAV
        path = f"sounds/{name}.wav"
        sf.write(path, clip.T.numpy(), sr)

        actual_duration = clip.shape[-1] / sr

        print(f"Saved: {path}")
        print(f"Duration: {actual_duration:.1f} seconds")
        display(Audio(path))

    except Exception as e:
        print(f"ERROR generating {name}: {e}")


print("\nGeneration process finished.")
print("Files currently saved in:", os.path.abspath("sounds"))
print("Number of WAV files:", len([
    f for f in os.listdir("sounds") if f.endswith(".wav")
]))

In [ ]:
import shutil
import os

shutil.make_archive("fictional_animal_sounds", "zip", "sounds")

print("ZIP created successfully!")
print("Files:", len([
    f for f in os.listdir("sounds")
    if f.endswith(".wav")
]))

In [ ]:
from google.colab import files

files.download("fictional_animal_sounds.zip")